In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

In [3]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from controllers.directional_trading.zlemog import ZLEMAConfig
import datetime
from decimal import Decimal


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "1m"
total_amount_quote = 1000
max_executors_per_side = 1
take_profit = 0.37
stop_loss = 0.045
trailing_stop_activation_price = 0.02
trailing_stop_trailing_delta = 0.002
time_limit = 7200  # 2 hours
cooldown_time = 60  # 1 minute
leverage = 20
take_profit_order_type = 2

# ZLEMA specific parameters
source = "close"
enable_kalman_filter = True
period_fast = 15
period_medium = 16
period_slow = 85
show_cross = True

# Creating the instance of the configuration and the controller
config = ZLEMAConfig(
    id=f"zlem_{connector_name}_{interval}_{trading_pair}",
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    source=source,
    enable_kalman_filter=enable_kalman_filter,
    period_fast=period_fast,
    period_medium=period_medium,
    period_slow=period_slow,
    show_cross=show_cross,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
            activation_price=Decimal(trailing_stop_activation_price),
            trailing_delta=Decimal(trailing_stop_trailing_delta)
        ),
        time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2024, 1, 1).timestamp())
end = int(datetime.datetime(2025, 3, 1).timestamp())


backtesting_result = await backtesting.run_backtesting(config, start, end, "1m")

2025-04-05 23:12:22,318 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17d483fa0>
2025-04-05 23:12:22,324 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x108922fe0>, 117907.790073541)])']
connector: <aiohttp.connector.TCPConnector object at 0x17d4839a0>
2025-04-05 23:14:25,437 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17e289540>
2025-04-05 23:14:25,439 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x17e2528c0>, 118030.867322375)])']
connector: <aiohttp.connector.TCPConnector object at 0x17e289570>
2025-04-05 23:16:26,931 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17d4835b0>
2025-04-05 23:16:26,934 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseH

In [5]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()


Net PNL: $18668.54 (1866.85%) | Max Drawdown: $-196.12 (-19.69%)
Total Volume ($): 15146000.00 | Sharpe Ratio: 5.61 | Profit Factor: 1.58
Total Executors: 7573 | Accuracy Long: 0.51 | Accuracy Short: 0.61
Close Types: Take Profit: 0 | Stop Loss: 55 | Time Limit: 7256 |
             Trailing Stop: 262 | Early Stop: 0



In [6]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,86J7FcthegaM63E7D9oQWArM7d8Xe56YzYg2gZCTxcz2,1704089340,position_executor,1704095040,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '86J7FcthegaM63E7D9oQWArM7d8Xe56YzYg2gZ...,-0.0040537503829763503471195207339405897073447...,-4.0537503829763510410089111246634274721145629...,0.59999999999999997779553950749686919152736663...,2000.000000000000227373675443232059478759765625,False,False,"{'close_price': 3.5779, 'level_id': None, 'sid...",None,BUY
1,74RX6gQCULnJT9hg1oSDva9CYnh2MzY3GozVrAJyMGsd,1704095040,position_executor,1704101940,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '74RX6gQCULnJT9hg1oSDva9CYnh2MzY3GozVrA...,-0.0002366583750245781743795636664629000733839...,-0.2366583750245781769816488804281107150018215...,0.59999999999999997779553950749686919152736663...,2000,False,False,"{'close_price': 3.5766, 'level_id': None, 'sid...",None,SELL
2,83neLgm2LcHk4QNruivvdACGrhPH65DxxgZ2oe4A45Td,1704098640,position_executor,1704104760,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '83neLgm2LcHk4QNruivvdACGrhPH65DxxgZ2oe...,-0.0003479979839838381579578130420316028903471...,-0.3479979839838381328043226403679000213742256...,0.59999999999999997779553950749686919152736663...,2000,False,False,"{'close_price': 3.5723, 'level_id': None, 'sid...",None,BUY
3,6Uob2qCvNvAa3tgkwx8PPpBNpJFbW7yFfzLZi4FdmVcT,1704104760,position_executor,1704111840,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '6Uob2qCvNvAa3tgkwx8PPpBNpJFbW7yFfzLZi4...,0.00491465442432053103982703134988696547225117...,4.91465442432053123411606065928936004638671875,0.59999999999999997779553950749686919152736663...,2000,False,False,"{'close_price': 3.5526, 'level_id': None, 'sid...",None,SELL
4,8sv2syoJw9PYhymWndzpyjHTXpUQxJB2EJ2TnMimFhE8,1704106620,position_executor,1704113520,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '8sv2syoJw9PYhymWndzpyjHTXpUQxJB2EJ2TnM...,-0.0083300098025485651287436184020407381467521...,-8.3300098025485667108114284928888082504272460...,0.59999999999999997779553950749686919152736663...,2000.000000000000227373675443232059478759765625,False,False,"{'close_price': 3.5429, 'level_id': None, 'sid...",None,BUY


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [7]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [8]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT